In [5]:
import ray

2026-09-05 17:34:06,696	INFO util.py:155 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


In [ ]:
from pathlib import Path

REPO_ROOT = Path.cwd() if (Path.cwd() / "KB Articles.pdf").exists() else Path.cwd().parent
KB_DB_BASE_PATH = REPO_ROOT / "kb_db"


In [ ]:
import lancedb
import pyarrow as pa

kb_db = lancedb.connect(KB_DB_BASE_PATH)

knowledge_base_table = kb_db.open_table(
    "knowledge_base",
)

In [ ]:
print(knowledge_base_table.schema)

In [ ]:
import torch

chunks_ds = ray.data.from_arrow(knowledge_base_table.to_arrow())
chunks_ds.schema()

In [ ]:
import numpy as np

MODEL_ID = "jinaai/jina-embeddings-v5-text-small"


class TextMatchingEmbedder:
    def __init__(self, model_id: str = MODEL_ID):
        import torch
        from transformers import AutoModel

        if torch.cuda.is_available():
            device = torch.device("cuda")
        elif torch.backends.mps.is_available():
            device = torch.device("mps")
        else:
            device = torch.device("cpu")

        load_kwargs = {
            "trust_remote_code": True,
            "dtype": torch.bfloat16,
        }
        if device.type == "cuda":
            load_kwargs["_attn_implementation"] = "flash_attention_2"

        model = AutoModel.from_pretrained(model_id, **load_kwargs)
        self.model = model.to(device=device)
        self.model.eval()

    def __call__(self, batch):
        texts = ["" if text is None else str(text) for text in batch["text"].tolist()]
        embeddings = self.model.encode(texts=texts, task="text-matching")
        if hasattr(embeddings, "detach"):
            embeddings = embeddings.detach().cpu().float().numpy()
        embeddings = np.asarray(embeddings, dtype=np.float32)
        batch = batch.copy()
        batch["embedding"] = list(embeddings)
        return batch


embedded_ds = chunks_ds.map_batches(
    TextMatchingEmbedder,
    batch_format="pandas",
    batch_size=8,
    compute=ray.data.ActorPoolStrategy(size=1),
    fn_constructor_kwargs={"model_id": MODEL_ID},
    num_gpus=1 if torch.cuda.is_available() else 0,
)
embedded_ds.schema()

In [ ]:
embedded_df = embedded_ds.to_pandas()
print(embedded_df["embedding"].iloc[0].shape)
embedded_df.head()

In [ ]:
knowledge_base_table = kb_db.create_table(
    "knowledge_base",
    data=embedded_df,
    mode="overwrite",
)
print(knowledge_base_table.schema)